# EE-411 — Fundamentals of Inference and Learning
## Exercise Session 4 — Scikit-learn, model selection and regression trees

**Learning objectives**
- Use scikit-learn's estimator interface for classification and regression.
- Select model parameters by cross-validation and interpret training and test errors.
- Explore the bias–variance trade-off with k-nearest neighbors (k-NN).
- Fit regression trees and, optionally, a random forest on the ozone dataset.

**Prerequisites**
- Supervised learning, k-NN and the Gaussian mixture model from TP3.
- Python packages: NumPy, pandas, Matplotlib and scikit-learn.
- The supplied `LAozone.data` file in the same folder as this notebook. Run the notebook from the TP4 folder.

Run the provided worked examples in order. Complete cells marked `TODO` / `YOUR CODE HERE` and the corresponding answer cells. **Only Exercise 1 — Regression trees is evaluated.** The random-forest exercise is optional and unassessed.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import mean_squared_error

np.random.seed(0)

## Part 1 — Introduction to scikit-learn

This introduction is adapted from [Jake VanderPlas's scikit-learn tutorial](https://github.com/jakevdp/sklearn_tutorial/), which includes source and license information.

Scikit-learn provides a common interface to machine learning algorithms in Python. The code in this part consists of provided worked examples: run each cell and inspect the results.

### About scikit-learn

Scikit-learn provides implementations of common machine learning algorithms through a consistent application programming interface (API). It is built on NumPy and SciPy, which provide numerical and scientific computing tools for Python.

### Representation of data in scikit-learn

Machine learning is about creating models from data: for that reason, we'll start by
discussing how data can be represented in order to be understood by the computer.  We will also use Matplotlib to visualize the data.

Most machine learning algorithms implemented in scikit-learn expect data to be stored in a
**two-dimensional array or matrix**.  The arrays can be
either ``numpy`` arrays, or in some cases ``scipy.sparse`` matrices.
The size of the array is expected to be `[n_samples, n_features]`

- **n_samples:**   The number of samples: each sample is an item to process (e.g. classify).
  A sample can be a document, a picture, a sound, a video, an astronomical object,
  a row in database or CSV file,
  or whatever you can describe with a fixed set of quantitative traits.
- **n_features:**  The number of features or distinct traits that can be used to describe each
  item in a quantitative manner.  Features are generally real-valued, but may be boolean or
  discrete-valued in some cases.

The number of features must be fixed in advance. However it can be very high dimensional
(e.g. millions of features) with most of them being zeros for a given sample. This is a case
where `scipy.sparse` matrices can be useful, in that they are
much more memory-efficient than numpy arrays.

### Basic principles of machine learning

We will introduce scikit-learn's *estimator* object, then use it for supervised classification and regression. The same interface also supports unsupervised methods such as dimensionality reduction and clustering.

### The scikit-learn estimator object

Every algorithm is exposed in scikit-learn via an *estimator* object. For instance, import the linear regression estimator as follows:

In [ ]:
from sklearn.linear_model import LinearRegression

**Estimator parameters**: All the parameters of an estimator can be set when it is instantiated, and have suitable default values:

In [ ]:
model = LinearRegression()
print(model)

**Estimated Model parameters**: When data is *fit* with an estimator, parameters are estimated from the data at hand. All the estimated parameters are attributes of the estimator object ending by an underscore:

In [ ]:
x = np.arange(10)
y = 2 * x + 1

In [ ]:
print(x)
print(y)

In [ ]:
plt.plot(x, y, 'o')
plt.xlabel("x")
plt.ylabel("y")
plt.title("Linear data");

In [ ]:
# The input data for sklearn is 2D: (samples == 10 x features == 1)
X = x[:, np.newaxis]
print(X)
print(y)

In [ ]:
# fit the model on our data
model.fit(X, y)

In [ ]:
# underscore at the end indicates a fit parameter
print(model.coef_)
print(model.intercept_)

The model found a line with a slope 2 and intercept 1, as we'd expect.

### Supervised learning: classification and regression

In **Supervised Learning**, we have a dataset consisting of both features and labels.
The task is to construct an estimator which is able to predict the label of an object
given the set of features. A relatively simple example is predicting the species of 
iris given a set of measurements of its flower. This is a relatively simple task. 
Some more complicated examples are:

- given a multicolor image of an object through a telescope, determine
  whether that object is a star, a quasar, or a galaxy.
- given a photograph of a person, identify the person in the photo.
- given a list of movies a person has watched and their personal rating
  of the movie, recommend a list of movies they would like
  (So-called *recommender systems*: a famous example is the Netflix Prize).

What these tasks have in common is that there is one or more unknown
quantities associated with the object which needs to be determined from other
observed quantities.

Supervised learning is further broken down into two categories, **classification** and **regression**.
In classification, the label is discrete, while in regression, the label is continuous. For example,
in astronomy, the task of determining whether an object is a star, a galaxy, or a quasar is a
classification problem: the label is from three distinct categories. On the other hand, we might
wish to estimate the age of an object based on such observations: this would be a regression problem,
because the label (age) is a continuous quantity.

---

### Regression Example

One of the simplest regression problems is fitting a line to data, which we saw above.
Scikit-learn also contains more sophisticated regression algorithms

In [ ]:
# Create some simple data
import numpy as np
np.random.seed(0)
X = np.random.random(size=(20, 1))
y = 2 * X.squeeze() + 1 + np.random.randn(20)

plt.plot(X.squeeze(), y, 'o')
plt.xlabel("x")
plt.ylabel("y")
plt.title("Noisy regression data");

As above, we can plot a line of best fit:

In [ ]:
model = LinearRegression()
model.fit(X, y)

# Plot the data and the model prediction
X_fit = np.linspace(0, 1, 100)[:, np.newaxis]
y_fit = model.predict(X_fit)

plt.plot(X.squeeze(), y, 'o', label='Data')
plt.plot(X_fit.squeeze(), y_fit, label='Prediction')
plt.legend()
plt.xlabel("x")
plt.ylabel("y")
plt.title("Linear regression fit");

Scikit-learn also has some more sophisticated models, which can respond to finer features in the data:

In [ ]:
# Fit a Random Forest
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(random_state=0)
model.fit(X, y)

# Plot the data and the model prediction
X_fit = np.linspace(0, 1, 100)[:, np.newaxis]
y_fit = model.predict(X_fit)

plt.plot(X.squeeze(), y, 'o', label='Data')
plt.plot(X_fit.squeeze(), y_fit, label='Prediction')
plt.legend()
plt.xlabel("x")
plt.ylabel("y")
plt.title("Random forest regression fit");

Whether either of these is a "good" fit or not depends on a number of things; we'll discuss details of how to choose a model later in the tutorial.

### Worked example — k-NN classification

The following provided code generates data from the Gaussian mixture model (GMM) introduced in TP3 and fits a k-NN classifier with $k=10$. Run it to compare training and test errors and visualize the decision boundary.

See the [KNeighborsClassifier documentation](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html) for the estimator used below.

In [ ]:
np.random.seed(0)
from sklearn import neighbors
clf = neighbors.KNeighborsClassifier(10)
# Samples 10 centroids for each class from two different bivariate Normal distributions
centroids_per_class = 10

class0_centroids = [1, 0] + np.random.randn(centroids_per_class, 2)
class1_centroids = [0, 1] + np.random.randn(centroids_per_class, 2)
samples_per_class = 100

# Sample actual data sampling from Normal distributions positioned around the centroids (Pick the centroid of the chosen class uniformly)
class0_labels = np.random.randint(centroids_per_class, size = samples_per_class)
class1_labels = np.random.randint(centroids_per_class, size = samples_per_class)

class0_samples = class0_centroids[class0_labels, :] + np.sqrt(1. / 5) * np.random.randn(samples_per_class, 2)
class1_samples = class1_centroids[class1_labels, :] + np.sqrt(1. / 5) * np.random.randn(samples_per_class, 2)

# Plot data
plt.figure(figsize=((12,5)))
plt.plot(class0_samples[:, 0], class0_samples[:, 1], "o", markersize=8, label="Class 0")
plt.plot(class1_samples[:, 0], class1_samples[:, 1], "x", markersize=8, label="Class 1")
plt.title("GMM training samples")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
X = np.vstack((class0_samples, class1_samples))
y = np.hstack((np.zeros(samples_per_class), np.ones(samples_per_class)))
clf.fit(X, y)

The classifier is now fitted. Next, generate independent test data from the same mixture model and compare the training and test errors.

In [ ]:
# Sample test data from the model
testsamples_per_class = 10000

class0_testlabels = np.random.randint(centroids_per_class, size = testsamples_per_class)
class1_testlabels = np.random.randint(centroids_per_class, size = testsamples_per_class)
class0_testsamples = class0_centroids[class0_testlabels, :] + np.sqrt(1. / 5) * np.random.randn(testsamples_per_class, 2)
class1_testsamples = class1_centroids[class1_testlabels, :] + np.sqrt(1. / 5) * np.random.randn(testsamples_per_class, 2)

X_test = np.vstack((class0_testsamples, class1_testsamples))
y_test = np.hstack((np.zeros(testsamples_per_class), np.ones(testsamples_per_class)))

In [ ]:
train_error = np.mean(y != clf.predict(X))
test_error = np.mean(y_test != clf.predict(X_test))

print("train/test error (for k = 10): %g/%g" % (train_error, test_error))

For classification, `.score()` returns accuracy, so the misclassification error is `1 - score`.

In [ ]:
train_error = 1. - clf.score(X, y)
test_error = 1. - clf.score(X_test, y_test)

print("train/test error (for k = 10): %g/%g" % (train_error, test_error))

To visualize the decision boundary, generate a grid and predict the class at each grid point.

In [ ]:
# Create a grid
xx, yy = np.meshgrid(np.linspace(-5, 5, 101), np.linspace(-5, 5, 101))
zz = clf.predict(np.c_[xx.ravel(), yy.ravel()])

# Do some plotting
zz = zz.reshape(xx.shape)
plt.pcolormesh(xx, yy, zz, cmap = "coolwarm", alpha = 0.2)
plt.plot(X[y == 0, 0], X[y == 0, 1], "o", label="Class 0", markersize=8)
plt.plot(X[y == 1, 0], X[y == 1, 1], "x", label="Class 1", markersize=8)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.title("k-NN decision boundary");

The same `fit` and `predict` interface applies to many other estimators.

### Recap: Scikit-learn's estimator interface

Scikit-learn strives to have a uniform interface across all methods,
and we'll see examples of these below. Given a scikit-learn *estimator*
object named `model`, the following methods are available:

- Available in **all Estimators**
  + `model.fit()` : fit training data. For supervised learning applications,
    this accepts two arguments: the data `X` and the labels `y` (e.g. `model.fit(X, y)`).
    For unsupervised learning applications, this accepts only a single argument,
    the data `X` (e.g. `model.fit(X)`).
- Available in **supervised estimators**
  + `model.predict()` : given a trained model, predict the label of a new set of data.
    This method accepts one argument, the new data `X_new` (e.g. `model.predict(X_new)`),
    and returns the learned label for each object in the array.
  + `model.predict_proba()` : For classification problems, some estimators also provide
    this method, which returns the probability that a new observation has each categorical label.
    In this case, the label with the highest probability is returned by `model.predict()`.
  + `model.score()` : the default score depends on the estimator. Classifiers usually return
    accuracy (between 0 and 1); regressors usually return the coefficient of determination
    $R^2$, which can be negative. Larger scores indicate a better fit.
- Available in **unsupervised estimators**
  + `model.predict()` : predict labels in clustering algorithms.
  + `model.transform()` : given an unsupervised model, transform new data into the new basis.
    This also accepts one argument `X_new`, and returns the new representation of the data based
    on the unsupervised model.
  + `model.fit_transform()` : some estimators implement this method,
    which more efficiently performs a fit and a transform on the same input data.

### Determining the best value of $k$ by cross-validation

Let us introduce a way to pinpoint the "best" value of $k$, called **cross-validation**.
The main idea is to partition our dataset into two subsets called *training* and *validation* sets. 

We can pinpoint the optimal $k$ so as to minimize the validation error.


**Caveat**: We then need a 3rd set where we can compute the test error -- **NEVER** adjust your parameters and compute the test error using the same set!

However, this does not provide us good statistics since the partitioning is only done once; a perhaps better way of doing it is by partitioning the dataset again and again, and recomputing the score. 

This is the idea behind cross-validation and scikit-learn has nice convenience functions to perform cross-validation for us.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Set up the grid search
parameters = [{'n_neighbors': np.arange(1, 20)}]
clf = GridSearchCV(neighbors.KNeighborsClassifier(), parameters, cv=5)
clf.fit(X, y)

# Print results
print(clf.best_params_)
print(clf.cv_results_["mean_test_score"])

Let's see how this does on the test set.

In [ ]:
train_error = np.mean(y != clf.predict(X))
test_error = np.mean(y_test != clf.predict(X_test))
print("train/test error (for optimal k): %g/%g" % (train_error, test_error))

### Choosing an estimator

The [scikit-learn estimator map](https://scikit-learn.org/stable/machine_learning_map.html) summarizes which algorithms to consider for different tasks.

Use this map as a starting point, then compare suitable estimators by cross-validation.

## Part 2 — The bias–variance trade-off

We now study how the training and test errors of k-NN change with model complexity.

### k-NN model complexity

Regenerate the training dataset and generate independent test data from the GMM used in TP3.

In [ ]:
np.random.seed(0)
# Samples 10 centroids for each class from two different bivariate Normal distributions
centroids_per_class = 10

class0_centroids = [1, 0] + np.random.randn(centroids_per_class, 2)
class1_centroids = [0, 1] + np.random.randn(centroids_per_class, 2)
samples_per_class = 100

# Sample actual data sampling from Normal distributions positioned around the centroids (Pick the centroid of the chosen class uniformly)
class0_labels = np.random.randint(centroids_per_class, size = samples_per_class)
class1_labels = np.random.randint(centroids_per_class, size = samples_per_class)

class0_samples = class0_centroids[class0_labels, :] + np.sqrt(1. / 5) * np.random.randn(samples_per_class, 2)
class1_samples = class1_centroids[class1_labels, :] + np.sqrt(1. / 5) * np.random.randn(samples_per_class, 2)

# Plot data
plt.figure(figsize=((12,5)))
plt.plot(class0_samples[:, 0], class0_samples[:, 1], "o", markersize=8, label="Class 0")
plt.plot(class1_samples[:, 0], class1_samples[:, 1], "x", markersize=8, label="Class 1")
plt.title("GMM training samples")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
X = np.vstack((class0_samples, class1_samples))
y = np.hstack((np.zeros(samples_per_class), np.ones(samples_per_class)))

In [ ]:
# Sample test data from the model
testsamples_per_class = 10000

class0_testlabels = np.random.randint(centroids_per_class, size = testsamples_per_class)
class1_testlabels = np.random.randint(centroids_per_class, size = testsamples_per_class)
class0_testsamples = class0_centroids[class0_testlabels, :] + np.sqrt(1. / 5) * np.random.randn(testsamples_per_class, 2)
class1_testsamples = class1_centroids[class1_testlabels, :] + np.sqrt(1. / 5) * np.random.randn(testsamples_per_class, 2)

X_test = np.vstack((class0_testsamples, class1_testsamples))
y_test = np.hstack((np.zeros(testsamples_per_class), np.ones(testsamples_per_class)))

In [ ]:
# Plot test data
plt.plot(X_test[y_test == 0, 0], X_test[y_test == 0, 1], "o", label="Class 0", ms=8)
plt.plot(X_test[y_test == 1, 0], X_test[y_test == 1, 1], "x", label="Class 1", ms=8)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.title("GMM test samples");

The following provided function computes the training and test misclassification errors of a k-NN classifier.

In [ ]:
from sklearn import neighbors
def knn_errors(X_train, y_train, X_test, y_test, k):
    clf = neighbors.KNeighborsClassifier(k)
    clf.fit(X_train, y_train)
    
    train_error = np.mean(y_train != clf.predict(X_train))
    test_error = np.mean(y_test != clf.predict(X_test))

    # Or use the score method 
    train_error = 1. - clf.score(X_train, y_train)
    test_error = 1. - clf.score(X_test, y_test)

    return train_error, test_error

It is instructive to look at the dependence of the error as a function of another quantity instead of $k$: the approximate **effective degrees of freedom** $N / k$, where $N$ is the training sample size. 

Indeed, the larger the $k$, the smaller the number of effective parameters -- think for instance of the $k = N$ limit, where everyone is assigned the same label.

In [ ]:
# Re-run the experiment above for a different range of values
ks = np.r_[np.arange(1, 10), np.arange(10, 150, 30)]
train_error = []
test_error = []
for k in ks:
    knn_errs = knn_errors(X, y, X_test, y_test,k)
    train_error.append(knn_errs[0])
    test_error.append(knn_errs[1])
    print("k = %d; train error = %g, test error = %g" % (k, train_error[-1], test_error[-1]))

# Plot error as a function of the degrees of freedom
plt.plot(len(y) / np.array(ks), train_error, "-o", label = "train")
plt.plot(len(y) / np.array(ks), test_error, "-o", label = "test")
plt.legend()
plt.xlabel(r"degrees of freedom $N / k$")
plt.ylabel("misclassification error")
plt.title("k-NN training and test errors");

The curves illustrate the **bias–variance trade-off**: increasing model complexity typically lowers the training error, while the test error can rise when the model overfits. The test curve need not be monotonic. Here it is shown for illustration; use validation data, rather than the test curve, to select $k$.

## Part 3 — Regression trees on the ozone dataset

### The ozone dataset

We will predict daily ozone concentration from weather measurements and the day of the year.


**Description**

These data record the level of atmospheric ozone concentration from
eight daily meteorological measurements made in the Los Angeles basin
in 1976.  Although measurements were made every day that year, some
observations were missing; here we have the 330 complete cases.  The target, `ozone`, is the supplied Upland maximum ozone measurement. We use the values as provided, without applying a log transform.

Source: [The Elements of Statistical Learning datasets](https://web.stanford.edu/~hastie/ElemStatLearn/datasets/LAozone.data). A copy is supplied as `LAozone.data`.

Detailed variable names:

- ozone : Upland Maximum Ozone
- vh : Vandenberg 500 mb Height
- wind : Wind Speed (mph)
- humidity : Humidity (%)
- temp : Sandburg AFB Temperature
- ibh : Inversion Base Height
- dpg : Daggot Pressure Gradient
- ibt : Inversion Base Temperature
- vis : Visibility (miles)
- doy : Day of the Year

Load the supplied dataset into a pandas DataFrame and inspect its first rows.

In [ ]:
data = pd.read_csv('LAozone.data')
print(f"data: {data.shape[0]} rows and {data.shape[1]} columns")
data.head()

The dataset contains

- $N=330$ data points $x_i$, each containing $P=9$ predictors (eight weather measurements and the day of the year). 

- One target $y_i$, the ozone concentration.

Now, before we touch anything, we need to follow best practices. When faced with a new dataset, we need to set up some kind of objective comparison. To do this, we need to split our dataset into three parts: **Training** (and within that, **Validation**), and **Testing** sets. 

The best practice here is to take the test data and lock it away somewhere. It is always tempting to tune your algorithms to give the best test performance. However, even if the regression isn't explicitly *trained* on the test data, as practitioners, we could be continually making changes in an effort to get our numbers up.

Instead, we should deep-freeze the test data, and then tune as much as we can via **cross-validation (CV)** on our training data.

In [ ]:
from sklearn.model_selection import train_test_split

# Convert from DataFrame to array
y = data['ozone'].values
feature_names = data.drop(columns='ozone').columns
X = data[feature_names].astype(float).values

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state=1)

print("Training samples: ", len(y_train))
print("Testing samples: ", len(y_test))

The following cell illustrates centering and scaling with `StandardScaler`. Fit preprocessing only on training data, then apply the same transformation to the test set. Trees do not require feature standardization; it is retained here to demonstrate the API. For scale-sensitive models, preprocessing must also be fitted separately within each cross-validation training fold.

We also subtract the training target mean, `ymean`, and add it back when plotting ozone predictions.

In [ ]:
from sklearn import preprocessing 

# Center and scale features and observations
scaler = preprocessing.StandardScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)
ymean = np.mean(y_train)
y_train = y_train - ymean
y_test = y_test - ymean

We will first fit a regression tree, then optionally compare it with an ensemble of trees.

### Exercise 1 — Regression trees (evaluated)

A regression tree recursively splits the training data using a feature threshold, such as $x_\mu < q$. Each split reduces a loss such as mean squared error (MSE) or mean absolute error (MAE). A new sample receives the mean target value of the training samples in its leaf for MSE, or the median for MAE.

Use [DecisionTreeRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeRegressor.html) on the ozone data:

1. Choose a parameter grid for at least two of `max_depth`, `min_samples_split` and `min_samples_leaf`. Set `random_state` for reproducibility.
2. Use five-fold cross-validation on the training set to select parameters, with `scoring="neg_mean_squared_error"`. Scikit-learn maximizes scores, so negate this score to report MSE.
3. Report the selected parameters, mean validation MSE, training MSE and test MSE. Plot predicted versus observed ozone values with labelled axes and an identity line. Use the test set only for this final evaluation.
4. Discuss the error differences and what they suggest about generalization and overfitting. Explain how the parameters control tree complexity.

In [ ]:
# TODO: tune a regression tree by cross-validation, report MSEs and plot predictions.
# YOUR CODE HERE

**Discussion — YOUR ANSWER HERE**

Write your interpretation of the results here.


## Bonus — Ensemble methods (optional, unassessed)

Ensemble methods combine predictions from several models. Here we combine regression trees to reduce the variance of their predictions.

### Bagging and random forests

In **bagging** (bootstrap aggregating), each tree is trained on a bootstrap sample: draw $N$ training observations with replacement, where $N$ is the training set size. Regression predictions are averaged over the trees.

Random forests additionally randomize the candidate features at each split. The `max_features` parameter controls how many features are considered. In scikit-learn's regressor, the default `max_features=1.0` considers all features; use `max_features="sqrt"` here to select a smaller random subset at each split.

For this optional example, use fixed parameter choices rather than another grid search.

### Optional exercise — Random forest regression

Fit a [RandomForestRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html) to the ozone training data using `n_estimators=100`, `max_features="sqrt"` and a fixed `random_state`. Report training and test MSE, plot predicted versus observed ozone values, and plot the feature importances with feature names.

Compare the errors with the regression tree if you completed Exercise 1. Verify that the forest prediction is the mean of its individual trees' predictions (`estimators_`), and briefly interpret the results.

In [ ]:
# TODO: fit a random forest, report MSEs, plot predictions and feature importances,
# and verify that its prediction is the average of the trees' predictions.
# YOUR CODE HERE

**Discussion — YOUR ANSWER HERE**

Write your interpretation of the results here.
